# Lesson 23 — Generalized Disjunctive Programming

## Learning objectives

1. Write a GDP with disjunctions over algebraic constraints.
2. Convert GDP to MINLP via big-M and convex-hull reformulations.
3. Recognize when convex hull is worth the extra variables.
4. Use `discopt`'s GDP frontend (or build one).

## 1. Disjunctive programming

A *disjunction* is an "or": at least one branch must hold. Standard form {cite:p}`Raman1994,Grossmann2013`:

$$
\bigvee_{k \in K} \begin{bmatrix} Y_k \\ A_k x \le b_k \\ f_k(x) \le 0 \end{bmatrix}
$$

with $Y_k$ a Boolean variable. Exactly one $Y_k$ is True. The constraints in branch $k$ are active iff $Y_k$ is True.

Logical constraints between $Y_k$'s can be expressed (e.g., $Y_1 \lor Y_2 \Rightarrow \neg Y_3$).

GDP is *the* natural framework for process synthesis problems with discrete unit-selection decisions {cite:p}`Grossmann2013,Trespalacios2015`.

## 2. Big-M reformulation

Convert each disjunctive constraint $A_k x \le b_k$ to

$$A_k x \le b_k + M_k (1 - y_k)$$

with binary $y_k$ corresponding to $Y_k$, and $\sum_k y_k = 1$. Plus indicators on $f_k(x) \le 0$.

**Pros:** simple, smaller model.
**Cons:** weak relaxation; tightness depends on $M_k$ {cite:p}`Sawaya2005`.

## 3. Convex-hull reformulation

For each $k$, introduce a copy $x_k$ of the variables. Let $x = \sum_k x_k$. Constrain $A_k x_k \le b_k y_k$ and $\sum_k y_k = 1$, $y_k \in [0, 1]$. The shorthand $x_k \in y_k X_k$ unfolds into the explicit **bound-scaling** constraints $\underline x \, y_k \le x_k \le \overline x \, y_k$, which force $x_k = 0$ whenever $y_k = 0$. The convex hull of the disjunction is then:

$$x = \sum_k x_k, \quad A_k x_k \le b_k y_k, \quad \underline x \, y_k \le x_k \le \overline x \, y_k, \quad \sum_k y_k = 1, \; y \ge 0.$$

For nonlinear convex $f_k$, the natural extension uses the **perspective function**: for a convex $f$ on a domain $D$, the perspective $\tilde f(x, y) = y f(x / y)$ for $y > 0$ (with $\tilde f(0, 0) = 0$ and $\tilde f(x, 0) = +\infty$ otherwise) is jointly convex in $(x, y)$ and provides the tightest convex underestimator of "$y = 1 \Rightarrow f(x) \le t$, $y = 0 \Rightarrow x = 0$" {cite:p}`Trespalacios2015`. We meet it again systematically in Lesson 29.

**Pros:** tightest possible LP relaxation.
**Cons:** $|K|$ copies of variables (model size grows). Worth it when big-M is loose.

## 4. Practical recipe

Default: big-M with carefully derived $M$. Switch to convex hull when:

- big-M values are "large" (no natural physical bound),
- the problem is solving slowly with much branching on indicators,
- the disjunction has a small number of branches ($|K| \le 3$).

In `discopt`, the GDP-style frontend (`Disjunct` / `add_disjunction`) is still maturing; today most users build the chosen reformulation **manually** (see the reactor cell below for big-M, and Exercises 1 and 2 for hull). {cite:p}`Sawaya2005`.

In [ ]:
# Standard discopt course imports. The lessons target the real
# `discopt.modeling.core.Model` API: `m.continuous(name, shape=..., lb=..., ub=...)`,
# `m.binary(name, shape=...)`, `m.integer(name, shape=..., lb=..., ub=...)`,
# `m.subject_to(...)`, `m.minimize(...) / .maximize(...)`, `m.solve(...)`.
# Result attributes: `r.status`, `r.objective`, `r.gap`, `r.bound`,
# `r.wall_time`, `r.node_count`, and `r.value(var)` for variable arrays.
import numpy as np
import discopt as do
import discopt.modeling as dm


In [ ]:
# Process synthesis: pick reactor type (CSTR or PFR) for one stage.
# CSTR: cost 100 + 5*F   (F flow)
# PFR : cost 60  + 8*F + 0.5*F^2
# Production target F = 4. Minimize cost.
#
# discopt's high-level GDP API takes Disjunct objects (see discopt.modeling.Disjunct).
# Below we implement the same problem manually with the big-M reformulation —
# this works against the stable real API today and is what the GDP frontend
# emits internally.
import numpy as np, discopt as do
from discopt.modeling import Model

m = Model("reactor")
F = m.continuous("F", lb=0, ub=10)
y_cstr = m.binary("y_cstr"); y_pfr = m.binary("y_pfr")
cost = m.continuous("cost", lb=0, ub=1000)
M = 1000

m.subject_to(F == 4)
m.subject_to(y_cstr + y_pfr == 1)                              # exactly one
# CSTR active: cost ≥ 100 + 5F  (and ≤ same, when active)
m.subject_to(cost >= 100 + 5 * F - M * (1 - y_cstr))
m.subject_to(cost <= 100 + 5 * F + M * (1 - y_cstr))
# PFR active
m.subject_to(cost >= 60 + 8 * F + 0.5 * F**2 - M * (1 - y_pfr))
m.subject_to(cost <= 60 + 8 * F + 0.5 * F**2 + M * (1 - y_pfr))
m.minimize(cost)
r = m.solve()
print("chosen reactor:", "CSTR" if r.value(y_cstr) > 0.5 else "PFR")
print("cost          :", r.objective)


## 5. Beyond two-branch disjunctions

Multi-branch disjunctions (3 or more options) and *nested* disjunctions are common in process synthesis. The convex-hull reformulation generalizes; big-M does too but loses tightness.

Logical constraints between disjunctions (e.g., "if reactor type A is chosen at stage 1 then separator type X cannot be chosen at stage 2") are easy to express as Boolean implications and reformulate to linear constraints on $y$.

## References
{cite:p}`Raman1994,Grossmann2013,Trespalacios2015,Sawaya2005`.